# CredChain Python — Document Extraction & Comparison Pipeline

A prototyping notebook for document text extraction, ID extraction, and semantic similarity comparison — all powered by Gemini.

**Capabilities:**  
- Extract raw text from documents (Gemini)  
- Extract document IDs / registration numbers (Gemini)  
- Compute embeddings (EmbeddingGemma)  
- Pairwise similarity + verdict (cosine similarity)  

**Verdict thresholds:** `tampered` (>= 0.95) | `suspicious` (>= 0.75) | `low_similarity` (>= 0.40) | `not_similar` (< 0.40)

> **Note:** This notebook uses Google Gemini API (cloud-based). The production `CredChain_Python` service uses offline models (PyMuPDF + EasyOCR + LaBSE).

In [ ]:
# Install dependencies: Gemini SDK, sentence-transformers, Pillow, NumPy
!pip install google-genai sentence-transformers Pillow numpy

## Setup — Imports, Configuration & Model Loading

Initialize the Gemini client and load the EmbeddingGemma model from Hugging Face.

In [ ]:
# ── Imports ──────────────────────────────────────────────
import io, json, time
import numpy as np
from google import genai
from google.genai import types
from google.colab import userdata
from huggingface_hub import login
from sentence_transformers import SentenceTransformer

# ── Configuration ────────────────────────────────────────────────
# Set in Colab via Tools → Secrets → Add secret
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
HF_TOKEN = userdata.get('HF_TOKEN')

EXTRACTION_MODEL = "gemini-3.1-flash-lite"
EMBEDDING_MODEL_ID = "google/embeddinggemma-300M"
RETRY_WAIT_SECONDS = 60

# ── Initialize clients ───────────────────────────────────
client = genai.Client(api_key=GEMINI_API_KEY)
login(token=HF_TOKEN)

print(f"Loading {EMBEDDING_MODEL_ID}...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_ID)
print("Ready — extraction via Gemini, embeddings via EmbeddingGemma")

Loading google/embeddinggemma-300M...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

Ready — extraction via Gemini, embeddings via EmbeddingGemma


## File Upload — Gemini Files API

Upload documents to Gemini's Files API so they can be referenced by the extraction step.
Polls until each file reaches `ACTIVE` state.

`_upload_files` is internal — use `batch_extract_documents` or `batch_extract_ids` instead.

In [ ]:
# ── MIME type detection ──────────────────────────────────
MIME_MAP = {
    ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
    ".png": "image/png",   ".tiff": "image/tiff",
    ".webp": "image/webp", ".pdf": "application/pdf",
}

def _upload_files(file_dict):
    """
    Upload files to Gemini Files API and wait until each is ACTIVE.

    Internal — called by _batch_extract.

    Args:
        file_dict: {filename: bytes_content, ...}
    Returns:
        list of (filename, File) tuples
    """
    uploaded = []
    for name, data in file_dict.items():
        ext = "." + name.rsplit(".", 1)[-1].lower() if "." in name else ""
        mime = MIME_MAP.get(ext, "application/octet-stream")

        f = client.files.upload(
            file=io.BytesIO(data),
            config=types.UploadFileConfig(mime_type=mime, display_name=name),
        )
        print(f"  Uploaded '{name}' ({f.state})")

        # Poll until ACTIVE or FAILED
        while True:
            info = client.files.get(name=f.name)
            if info.state == "ACTIVE":
                break
            if info.state == "FAILED":
                raise RuntimeError(f"File '{name}' failed processing")
            time.sleep(1)

        uploaded.append((name, f))
    return uploaded

## Document Extraction — Gemini Flash

Call Gemini to extract content from each document.

**Internal helpers:**  
- `_extract_document` — single Gemini call, no retry  
- `_extract_document_with_retry` — wraps `_extract_document` with rate-limit retry (HTTP 429)  
- `_batch_extract(prompt)` — shared pipeline: upload → extract each file → return `[(filename, raw_dict), ...]`  

**Public:**  
- `batch_extract_documents` — extract raw text + IDs (for embedding + search)  
- `batch_extract_ids` — extract document IDs / registration numbers

In [ ]:
def _extract_document(contents):
    """
    Call Gemini to process a document.
    Returns parsed JSON dict, or {} on empty/non-JSON response.
    """
    response = client.models.generate_content(
        model=EXTRACTION_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        ),
    )
    if not response.text:
        return {}
    try:
        return json.loads(response.text)
    except json.JSONDecodeError:
        print(f"  Warning: Gemini returned non-JSON: {response.text[:200]}")
        return {}


def _extract_document_with_retry(contents, max_retries=3):
    """
    Call _extract_document with retry on rate limit (HTTP 429 / RESOURCE_EXHAUSTED).
    """
    for attempt in range(max_retries):
        try:
            return _extract_document(contents)
        except Exception as e:
            msg = str(e)
            if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                print(f"  Rate limited — retrying in {RETRY_WAIT_SECONDS}s "
                      f"(attempt {attempt+1}/{max_retries})")
                time.sleep(RETRY_WAIT_SECONDS)
            else:
                raise
    raise RuntimeError("Max retries exceeded")


def _batch_extract(file_dict, prompt):
    """
    Upload files and extract via Gemini with retry. Shared internal pipeline.

    Returns list of (filename, raw_dict) tuples.
    """
    print(f"Uploading {len(file_dict)} file(s)...")
    uploaded = _upload_files(file_dict)

    results = []
    for name, f in uploaded:
        print(f"  Extracting '{name}'...")
        contents = [
            types.Part.from_uri(file_uri=f.uri, mime_type=f.mime_type),
            types.Part.from_text(text=prompt),
        ]
        try:
            raw = _extract_document_with_retry(contents)
            results.append((name, raw))
            print(f"    Done")
        except RuntimeError:
            print(f"    Failed after retries, skipping '{name}'")
        time.sleep(1)

    return results


# ── Prompts ──────────────────────────────────────────────

PROMPT_EXTRACT_DOCUMENT = (
    "Extract all textual content from this document. "
    "Omit headers, footers, boilerplate, and formatting artifacts. "
    "Also extract all document IDs, registration numbers, and identifier codes. "
    "For each ID, identify its type (e.g. passport, driver_license, tax_id, "
    "student_id, national_id, etc.). "
    "Return a JSON object with keys 'raw_text' (string) and 'ids' "
    "(array of {type: str, value: str} objects)."
)
PROMPT_EXTRACT_IDS = (
    "Extract all document IDs, registration numbers, and identifier codes "
    "from this document. "
    "For each ID, identify its type (e.g. passport, driver_license, tax_id, "
    "student_id, national_id, etc.). "
    "Return a JSON object with key 'ids' containing an array of "
    "{type: str, value: str} objects."
)


# ── Public API ───────────────────────────────────────────

def batch_extract_documents(file_dict):
    """
    Upload files and extract raw text + IDs via Gemini.

    Returns list of {"filename": str, "raw_text": str, "ids": [{type, value}, ...]}.
    """
    results = _batch_extract(file_dict, PROMPT_EXTRACT_DOCUMENT)
    return [
        {"filename": name, "raw_text": raw.get("raw_text", ""),
         "ids": raw.get("ids", [])}
        for name, raw in results
    ]


def batch_extract_ids(file_dict):
    """
    Upload files and extract document IDs via Gemini.

    Returns list of {"filename": str, "ids": [{"type": str, "value": str}, ...]}.
    """
    results = _batch_extract(file_dict, PROMPT_EXTRACT_IDS)
    return [
        {"filename": name, "ids": raw.get("ids", [])}
        for name, raw in results
    ]

## Embeddings & Similarity

Compute text embeddings via EmbeddingGemma, then measure cosine similarity between document pairs.

`compute_embedding` for single texts, `batch_compute_embeddings` for multiple.

**Verdict thresholds:**

| Verdict | Similarity |
|---|---|
| `tampered` | >= 0.95 |
| `suspicious` | >= 0.75 |
| `low_similarity` | >= 0.40 |
| `not_similar` | < 0.40 |

In [ ]:
def compute_embedding(text):
    """Encode a single text via EmbeddingGemma. Returns list of floats."""
    return embedding_model.encode(text).tolist()


def batch_compute_embeddings(texts):
    """Encode multiple texts via EmbeddingGemma. Returns list of float lists."""
    return embedding_model.encode(texts).tolist()


def cosine_similarity(a, b):
    """Cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def verdict_for(similarity):
    """Map similarity score to verdict label."""
    if similarity >= 0.95:
        return "tampered"
    if similarity >= 0.75:
        return "suspicious"
    if similarity >= 0.40:
        return "low_similarity"
    return "not_similar"

## Demo 1 — Document Extraction + Embedding

Upload one or more documents, extract raw text + IDs via Gemini, and compute embeddings.

In [ ]:
# ── Demo 1: Extract raw text and embed each document ──────
from google.colab import files

uploaded = files.upload()
docs = batch_extract_documents(uploaded)

for doc in docs:
    fname = doc["filename"]
    text = doc["raw_text"]
    emb = compute_embedding(text)

    print(f"\nFile: {fname}")
    print(f"  Text: {text[:200]}..." if len(text) > 200 else f"  Text: {text}")
    ids = doc["ids"]
    if ids:
        print(f"  IDs: {len(ids)} found")
        for item in ids:
            print(f"    [{item['type']}] {item['value']}")
    print(f"Embedding: {len(emb)} dimensions")

Saving 73 - Liesbeth Stifanny.pdf to 73 - Liesbeth Stifanny.pdf
Uploading 1 file(s)...
  Uploaded '73 - Liesbeth Stifanny.pdf' (FileState.ACTIVE)
  Extracting '73 - Liesbeth Stifanny.pdf'...
    Done

File: 73 - Liesbeth Stifanny.pdf
  Text: SERTIFIKAT PARTISIPASI Liesbeth Stifanny atas partisipasi dan kontribusinya yang tulus sebagai PESERTA dalam ajang Brawijaya Lomba Menulis Kisahku, Inspirasiku, yang bertujuan menumbuhkan kecintaan te...
  IDs: 1 found
    [registration_number] BLMKI/LDRS-2026/III/307
Embedding: 768 dimensions


## Demo 2 — ID Search

Phase 1: Upload documents to store (extracts raw text + IDs via Gemini).
Phase 2: Upload a single document to search — extracts its IDs and
matches against stored documents by exact ID value.


In [ ]:
# ── Demo 2: Two-phase ID search ───────────────────────────
from google.colab import files

# Phase 1 — Store
print("=== Phase 1: Store documents ===")
print("Upload 1 or more documents to store:")
uploaded = files.upload()
stored_docs = batch_extract_documents(uploaded)

for doc in stored_docs:
    print(f"\n  Stored: {doc['filename']}")
    print(f"    Text: {doc['raw_text'][:100]}...")
    print(f"    IDs: {len(doc['ids'])} found")
    for item in doc['ids']:
        print(f"      [{item['type']}] {item['value']}")

print(f"\nStored {len(stored_docs)} document(s) in memory.")

# Phase 2 — Search
print("\n=== Phase 2: Search ===")
print("Upload 1 document to search:")
search_uploaded = files.upload()
search_results = batch_extract_ids(search_uploaded)
search_ids = search_results[0]['ids']

# Match by exact value
matched_docs = []
for stored in stored_docs:
    stored_values = {item['value'] for item in stored['ids']}
    matched = [item for item in search_ids if item['value'] in stored_values]
    if matched:
        matched_docs.append((stored, matched))

if matched_docs:
    for stored, matched in matched_docs:
        print(f"\n  Match: {stored['filename']}")
        print(f"    IDs matched: {len(matched)}")
        for item in matched:
            print(f"      [{item['type']}] {item['value']}")
        print(f"    Text: {stored['raw_text'][:100]}...")

    # Unmatched search IDs
    all_matched_values = set()
    for _, matched in matched_docs:
        all_matched_values.update(item['value'] for item in matched)
    unmatched = [item for item in search_ids if item['value'] not in all_matched_values]
    if unmatched:
        print(f"\n  Unmatched search IDs: {len(unmatched)}")
        for item in unmatched:
            print(f"    [{item['type']}] {item['value']}")

else:
    print("\n  No stored documents matched the search IDs.")


=== Phase 1: Store documents ===
Upload 1 or more documents to store:


Saving 73 - Liesbeth Stifanny.pdf to 73 - Liesbeth Stifanny (1).pdf
Saving beths-high-school-diploman.jpg to beths-high-school-diploman.jpg
Saving English-beginner 2.jpg to English-beginner 2.jpg
Saving LIESBETH STIFANNY H.png to LIESBETH STIFANNY H.png
Saving SERTIFIKAT PKKMB 2025.pdf to SERTIFIKAT PKKMB 2025.pdf
Saving Sertifkat peserta LKTI - PKTJ.pdf to Sertifkat peserta LKTI - PKTJ.pdf
Uploading 6 file(s)...
  Uploaded '73 - Liesbeth Stifanny (1).pdf' (FileState.ACTIVE)
  Uploaded 'beths-high-school-diploman.jpg' (FileState.ACTIVE)
  Uploaded 'English-beginner 2.jpg' (FileState.ACTIVE)
  Uploaded 'LIESBETH STIFANNY H.png' (FileState.ACTIVE)
  Uploaded 'SERTIFIKAT PKKMB 2025.pdf' (FileState.ACTIVE)
  Uploaded 'Sertifkat peserta LKTI - PKTJ.pdf' (FileState.ACTIVE)
  Extracting '73 - Liesbeth Stifanny (1).pdf'...
    Done
  Extracting 'beths-high-school-diploman.jpg'...
    Done
  Extracting 'English-beginner 2.jpg'...
    Done
  Extracting 'LIESBETH STIFANNY H.png'...
    Done
  Ext

Saving 73 - Liesbeth Stifanny.pdf to 73 - Liesbeth Stifanny (2).pdf
Uploading 1 file(s)...
  Uploaded '73 - Liesbeth Stifanny (2).pdf' (FileState.ACTIVE)
  Extracting '73 - Liesbeth Stifanny (2).pdf'...
    Done

  Match: 73 - Liesbeth Stifanny (1).pdf
    IDs matched: 1
      [registration_number] BLMKI/LDRS-2026/III/307
    Text: SERTIFIKAT PARTISIPASI Liesbeth Stifanny atas partisipasi dan kontribusinya yang tulus sebagai PESER...


## Demo 3 — Pairwise Document Comparison

Upload at least 2 documents (reference + comparison). Extract raw text from both,
compute embeddings via `batch_compute_embeddings`, measure cosine similarity, and assign a verdict.

In [ ]:
# ── Demo 3: Compare two documents ─────────────────────────
from google.colab import files

print("Upload at least 2 files (reference + comparison):")
uploaded = files.upload()
if len(uploaded) < 2:
    print("Need at least 2 files. Upload more:")
    uploaded.update(files.upload())

docs = batch_extract_documents(uploaded)

if len(docs) < 2:
    print("Error: Could not extract from at least 2 files. "
          "Ensure you upload valid documents.")
else:
    ref, cmp = docs[0], docs[1]

    embeddings = batch_compute_embeddings([ref["raw_text"], cmp["raw_text"]])
    sim = cosine_similarity(embeddings[0], embeddings[1])

    print(f"\nReference: {ref['filename']}")
    print(f"  {ref['raw_text'][:200]}..." if len(ref['raw_text']) > 200 else f"  {ref['raw_text']}")
    print(f"\nComparison: {cmp['filename']}")
    print(f"  {cmp['raw_text'][:200]}..." if len(cmp['raw_text']) > 200 else f"  {cmp['raw_text']}")
    print(f"\nSimilarity: {sim:.4f}")
    print(f"Verdict: {verdict_for(sim)}")

Upload at least 2 files (reference + comparison):


Saving English-beginner 2.jpg to English-beginner 2 (1).jpg
Need at least 2 files. Upload more:


Saving Sertifkat peserta LKTI - PKTJ.pdf to Sertifkat peserta LKTI - PKTJ (1).pdf
Uploading 2 file(s)...
  Uploaded 'English-beginner 2 (1).jpg' (FileState.ACTIVE)
  Uploaded 'Sertifkat peserta LKTI - PKTJ (1).pdf' (FileState.ACTIVE)
  Extracting 'English-beginner 2 (1).jpg'...
    Done
  Extracting 'Sertifkat peserta LKTI - PKTJ (1).pdf'...
    Done

Reference: English-beginner 2 (1).jpg
  CERTIFICATE OF COMPLETION This certificate is proudly presented to Liesbeth Stifanny for successfully completing the 15-Day Basic English Program for Beginners 2 The participant has demonstrated the a...

Comparison: Sertifkat peserta LKTI - PKTJ (1).pdf
  SERTIFIKAT PENGHARGAAN 167/LKTI/PKTJ-FEST/2026 Diberikan kepada: Liesbeth Stifanny Sebagai: PESERTA Lomba Karya Tulis Ilmiah Nasional Tahun 2026 Kategori Mahasiswa Dalam rangka memperingati Dies Natal...

Similarity: 0.3391
Verdict: not_similar
